In [7]:
# ── Notebook 01b: Open Library — genre-first query ─────────────────────────
import requests, json, time, re
import pandas as pd
from pathlib import Path

REPO_ROOT     = Path().resolve().parents[1]
RAW_DIR       = REPO_ROOT / "Data" / "Raw" / "non_western_fantasy"
KEYWORDS_PATH = REPO_ROOT / "Data" / "Keywords" / "non_western_fantasy_keywords.json"
OUT_PATH      = RAW_DIR / "ol_genre_first.json"
CKPT_PATH     = RAW_DIR / "ol_genre_first_checkpoint.json"

# ── 1. Genre queries — OL returns only tagged fantasy/scifi ────────────────
# Each tuple is (ol_subject_query, label)
# OL subject search is AND between multiple subject= params
GENRE_SUBJECT_QUERIES = [
    # Africa
    ("fantasy+africa",               "africa"),
    ("fantasy+african",              "africa"),
    ("science+fiction+africa",       "africa"),
    ("fantasy+yoruba",               "yoruba"),
    ("fantasy+igbo",                 "igbo"),
    ("fantasy+akan",                 "akan"),
    ("fantasy+zulu",                 "zulu"),
    ("fantasy+orisha",               "orisha"),
    ("fantasy+anansi",               "anansi"),
    ("afrofuturism",                 "afrofuturism"),
    ("africanfuturism",              "afrofuturism"),
    # East Asia
    ("fantasy+chinese",              "chinese"),
    ("fantasy+japan",                "japanese"),
    ("fantasy+japanese",             "japanese"),
    ("fantasy+korean",               "korean"),
    ("wuxia",                        "wuxia"),
    ("xianxia",                      "xianxia"),
    ("fantasy+yokai",                "japanese"),
    ("fantasy+kitsune",              "japanese"),
    ("fantasy+cultivation",          "chinese"),
    # South / Southeast Asia
    ("fantasy+india",                "south_asian"),
    ("fantasy+indian",               "south_asian"),
    ("fantasy+hindu+mythology",      "south_asian"),
    ("fantasy+mahabharata",          "south_asian"),
    ("fantasy+ramayana",             "south_asian"),
    ("fantasy+philippine",           "filipino"),
    ("fantasy+filipino",             "filipino"),
    ("fantasy+aswang",               "filipino"),
    ("fantasy+vietnamese",           "southeast_asian"),
    ("fantasy+thai",                 "southeast_asian"),
    # Middle East / Persian
    ("fantasy+arabian",              "middle_eastern"),
    ("fantasy+djinn",                "middle_eastern"),
    ("fantasy+persian",              "middle_eastern"),
    ("fantasy+islamic",              "middle_eastern"),
    ("fantasy+ottoman",              "middle_eastern"),
    ("fantasy+mughal",               "south_asian"),
    # Latin America
    ("fantasy+aztec",                "latin_american"),
    ("fantasy+maya",                 "latin_american"),
    ("fantasy+mayan",                "latin_american"),
    ("fantasy+inca",                 "latin_american"),
    ("fantasy+latin+american",       "latin_american"),
    ("science+fiction+latin+american","latin_american"),
    ("fantasy+curandera",            "latin_american"),
    ("fantasy+mesoamerican",         "latin_american"),
    # Indigenous Americas
    ("fantasy+native+american",      "indigenous_americas"),
    ("fantasy+indigenous",           "indigenous_americas"),
    ("fantasy+navajo",               "indigenous_americas"),
    ("fantasy+lakota",               "indigenous_americas"),
    # Oceania
    ("fantasy+maori",                "oceania"),
    ("fantasy+aboriginal",           "oceania"),
    ("fantasy+pacific+islander",     "oceania"),
    ("fantasy+polynesian",           "oceania"),
]

DELAY      = 1.0   # seconds between requests
PAGE_LIMIT = 20    # max pages per query (20 results/page = 400 books max per query)

# ── 2. Cultural match keywords (for tagging, not filtering) ────────────────
# Load from your existing keywords JSON
KEYWORDS_PATH = REPO_ROOT / "Data" / "Keywords" / "non_western_fantasy_keywords.json"

with open(KEYWORDS_PATH) as f:
    KW = json.load(f)

# Flatten all find_keywords values into a single set for subject matching
def flatten_find_keywords(kw_dict):
    terms = set()
    for v in kw_dict.values():
        if isinstance(v, list):
            for t in v:
                terms.add(t.lower())
        elif isinstance(v, dict):
            terms |= flatten_find_keywords(v)
    return terms

CULTURAL_TERMS = flatten_find_keywords(KW["find_keywords"])

# ── 3. Fetcher ─────────────────────────────────────────────────────────────
def fetch_ol_page(subject_query, page=1):
    url = f"https://openlibrary.org/search.json"
    params = {
        "subject": subject_query.replace("+", " "),
        "fields": "title,author_name,first_publish_year,cover_i,ratings_average,ratings_count,subject,key",
        "limit": 100,
        "offset": (page - 1) * 100,
        "language": "eng",
    }
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json()

def ol_to_record(doc, source_tag, query):   # add query param
    cover_i = doc.get("cover_i")
    subjects = doc.get("subject", [])
    return {
        "title":          doc.get("title", "").strip(),
        "author":         doc.get("author_name", [""])[0] if doc.get("author_name") else "",
        "year_published": doc.get("first_publish_year"),
        "cover_url":      f"https://covers.openlibrary.org/b/id/{cover_i}-M.jpg" if cover_i else "",
        "avg_rating":     doc.get("ratings_average"),
        "num_ratings":    doc.get("ratings_count"),
        "subjects":       subjects,
        "source":         "open_library",
        "source_url":     f"https://openlibrary.org{doc.get('key', '')}",
        "source_tag":     source_tag,
        "query":          query,        # ← new
        "description":    "",
    }

# ── 4. Main scrape loop ────────────────────────────────────────────────────
# Resume from checkpoint
if CKPT_PATH.exists():
    with open(CKPT_PATH) as f:
        all_books = json.load(f)
    done_queries = {b["source_tag"] for b in all_books}  # rough resume
    print(f"Resuming — {len(all_books):,} books already collected")
else:
    all_books = []
    done_queries = set()

seen_keys = {b["source_url"] for b in all_books}   # dedup by OL key

for subject_query, tag in GENRE_SUBJECT_QUERIES:
    if subject_query in done_queries:      # ← now matches query string, not tag
        print(f"  ⏭  Skipping {subject_query} (already done)")
        continue

    print(f"\n── {subject_query} ({tag}) ──")
    page_num  = 1
    query_new = 0

    while page_num <= PAGE_LIMIT:
        try:
            data = fetch_ol_page(subject_query, page_num)
            docs = data.get("docs", [])
            if not docs:
                break

            for doc in docs:
                key = f"https://openlibrary.org{doc.get('key', '')}"
                if key in seen_keys:
                    continue
                seen_keys.add(key)
                rec = ol_to_record(doc, tag, subject_query)   # ← pass query
                if rec["title"]:
                    all_books.append(rec)
                    query_new += 1
            # ... rest stays the same

            print(f"  Page {page_num}: {len(docs)} results, {query_new} new total")

            if len(docs) < 100:   # last page
                break

            page_num += 1
            time.sleep(DELAY)

        except Exception as e:
            print(f"  ❌ Error on page {page_num}: {e}")
            time.sleep(5)
            break

    # ── Replace the checkpoint resume block ────────────────────────────────────
if CKPT_PATH.exists():
    with open(CKPT_PATH) as f:
        all_books = json.load(f)
    # Track by query string, not tag (tags are not unique per query)
    done_queries = {b.get("query") for b in all_books if b.get("query")}
    print(f"Resuming — {len(all_books):,} books, {len(done_queries)} queries done")
else:
    all_books = []
    done_queries = set()

seen_keys = {b["source_url"] for b in all_books}

print(f"\n✅ Done — {len(all_books):,} books total")

# ── 5. Save final ──────────────────────────────────────────────────────────
with open(OUT_PATH, "w") as f:
    json.dump(all_books, f, indent=2)
print(f"Saved to {OUT_PATH}")


── fantasy+africa (africa) ──
  Page 1: 100 results, 100 new total
  Page 2: 5 results, 105 new total

── fantasy+african (africa) ──
  Page 1: 100 results, 92 new total
  Page 2: 42 results, 131 new total

── science+fiction+africa (africa) ──
  Page 1: 77 results, 69 new total

── fantasy+yoruba (yoruba) ──
  Page 1: 2 results, 1 new total

── fantasy+igbo (igbo) ──
  Page 1: 1 results, 1 new total

── fantasy+akan (akan) ──

── fantasy+zulu (zulu) ──
  Page 1: 1 results, 0 new total

── fantasy+orisha (orisha) ──

── fantasy+anansi (anansi) ──
  Page 1: 7 results, 6 new total

── afrofuturism (afrofuturism) ──
  Page 1: 19 results, 19 new total

── africanfuturism (afrofuturism) ──

── fantasy+chinese (chinese) ──
  Page 1: 100 results, 97 new total
  Page 2: 29 results, 126 new total

── fantasy+japan (japanese) ──
  Page 1: 100 results, 99 new total
  Page 2: 100 results, 199 new total
  Page 3: 100 results, 299 new total
  Page 4: 31 results, 330 new total

── fantasy+japanese (

In [8]:
import json
from pathlib import Path

# Check checkpoint first
CKPT_PATH = Path(r"C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Data\Raw\non_western_fantasy\ol_genre_first_checkpoint.json")
OUT_PATH  = CKPT_PATH.parent / "ol_genre_first.json"

for label, path in [("Checkpoint", CKPT_PATH), ("Output", OUT_PATH)]:
    if path.exists():
        with open(path) as f:
            books = json.load(f)
        print(f"{label}: {len(books):,} books")
        print(f"  Sample: {books[0]['title']} — {books[0]['author']}")
    else:
        print(f"{label}: NOT FOUND")

Checkpoint: NOT FOUND
Output: 0 books


IndexError: list index out of range

In [10]:
# ── Clean re-run — fixed version ───────────────────────────────────────────
import requests, json, time
import pandas as pd
from pathlib import Path

REPO_ROOT  = Path().resolve().parents[1]
RAW_DIR    = REPO_ROOT / "Data" / "Raw" / "non_western_fantasy"
CKPT_PATH  = RAW_DIR / "ol_genre_first_checkpoint.json"
OUT_PATH   = RAW_DIR / "ol_genre_first.json"

# Clear broken checkpoints
CKPT_PATH.write_text("[]")
OUT_PATH.write_text("[]")

GENRE_SUBJECT_QUERIES = [
    ("fantasy africa",               "africa"),
    ("fantasy african",              "africa"),
    ("science fiction africa",       "africa"),
    ("fantasy yoruba",               "yoruba"),
    ("fantasy igbo",                 "igbo"),
    ("fantasy akan",                 "akan"),
    ("fantasy zulu",                 "zulu"),
    ("fantasy orisha",               "orisha"),
    ("fantasy anansi",               "anansi"),
    ("afrofuturism",                 "afrofuturism"),
    ("africanfuturism",              "afrofuturism"),
    ("fantasy chinese",              "chinese"),
    ("fantasy japan",                "japanese"),
    ("fantasy japanese",             "japanese"),
    ("fantasy korean",               "korean"),
    ("wuxia",                        "wuxia"),
    ("xianxia",                      "xianxia"),
    ("fantasy yokai",                "japanese"),
    ("fantasy kitsune",              "japanese"),
    ("fantasy cultivation",          "chinese"),
    ("fantasy india",                "south_asian"),
    ("fantasy indian",               "south_asian"),
    ("fantasy hindu mythology",      "south_asian"),
    ("fantasy mahabharata",          "south_asian"),
    ("fantasy ramayana",             "south_asian"),
    ("fantasy philippine",           "filipino"),
    ("fantasy filipino",             "filipino"),
    ("fantasy aswang",               "filipino"),
    ("fantasy vietnamese",           "southeast_asian"),
    ("fantasy thai",                 "southeast_asian"),
    ("fantasy arabian",              "middle_eastern"),
    ("fantasy djinn",                "middle_eastern"),
    ("fantasy persian",              "middle_eastern"),
    ("fantasy islamic",              "middle_eastern"),
    ("fantasy ottoman",              "middle_eastern"),
    ("fantasy mughal",               "south_asian"),
    ("fantasy aztec",                "latin_american"),
    ("fantasy maya",                 "latin_american"),
    ("fantasy mayan",                "latin_american"),
    ("fantasy inca",                 "latin_american"),
    ("fantasy latin american",       "latin_american"),
    ("science fiction latin american","latin_american"),
    ("fantasy curandera",            "latin_american"),
    ("fantasy mesoamerican",         "latin_american"),
    ("fantasy native american",      "indigenous_americas"),
    ("fantasy indigenous",           "indigenous_americas"),
    ("fantasy navajo",               "indigenous_americas"),
    ("fantasy lakota",               "indigenous_americas"),
    ("fantasy maori",                "oceania"),
    ("fantasy aboriginal",           "oceania"),
    ("fantasy pacific islander",     "oceania"),
    ("fantasy polynesian",           "oceania"),
]

PAGE_LIMIT = 20
DELAY      = 0.5

def fetch_ol_page(query, page=1):
    r = requests.get(
        "https://openlibrary.org/search.json",
        params={
            "q":      query,
            "fields": "title,author_name,first_publish_year,cover_i,ratings_average,ratings_count,subject,key",
            "limit":  100,
            "offset": (page - 1) * 100,
        },
        timeout=20
    )
    r.raise_for_status()
    return r.json()

def to_record(doc, tag, query):
    cover_i = doc.get("cover_i")
    return {
        "title":          doc.get("title", "").strip(),
        "author":         (doc.get("author_name") or [""])[0],
        "year_published": doc.get("first_publish_year"),
        "cover_url":      f"https://covers.openlibrary.org/b/id/{cover_i}-M.jpg" if cover_i else "",
        "avg_rating":     doc.get("ratings_average"),
        "num_ratings":    doc.get("ratings_count"),
        "subjects":       doc.get("subject", []),
        "source":         "open_library",
        "source_url":     f"https://openlibrary.org{doc.get('key', '')}",
        "source_tag":     tag,
        "query":          query,
        "description":    "",
    }

# ── Main loop ──────────────────────────────────────────────────────────────
all_books  = []
seen_keys  = set()
done_queries = set()

for query, tag in GENRE_SUBJECT_QUERIES:
    print(f"\n── {query} ({tag}) ──")
    page_num  = 1
    query_new = 0

    while page_num <= PAGE_LIMIT:
        try:
            data = fetch_ol_page(query, page_num)
            docs = data.get("docs", [])
            if not docs:
                break

            for doc in docs:
                key = f"https://openlibrary.org{doc.get('key', '')}"
                if key in seen_keys:
                    continue
                seen_keys.add(key)
                rec = to_record(doc, tag, query)
                if rec["title"]:
                    all_books.append(rec)
                    query_new += 1

            print(f"  Page {page_num}: {len(docs)} results | {query_new} new | total: {len(all_books):,}")

            if len(docs) < 100:
                break

            page_num += 1
            time.sleep(DELAY)

        except Exception as e:
            print(f"  ❌ Error: {e}")
            time.sleep(5)
            break

    # Save checkpoint after every query
    with open(CKPT_PATH, "w") as f:
        json.dump(all_books, f)

print(f"\n✅ Done — {len(all_books):,} books")

# Final save
with open(OUT_PATH, "w") as f:
    json.dump(all_books, f, indent=2)
print(f"Saved to {OUT_PATH}")


── fantasy africa (africa) ──
  Page 1: 100 results | 100 new | total: 100
  Page 2: 100 results | 200 new | total: 200
  Page 3: 27 results | 227 new | total: 227

── fantasy african (africa) ──
  Page 1: 100 results | 86 new | total: 313
  Page 2: 100 results | 171 new | total: 398
  Page 3: 73 results | 236 new | total: 463

── science fiction africa (africa) ──
  Page 1: 100 results | 83 new | total: 546
  Page 2: 95 results | 172 new | total: 635

── fantasy yoruba (yoruba) ──
  Page 1: 3 results | 1 new | total: 636

── fantasy igbo (igbo) ──
  Page 1: 4 results | 3 new | total: 639

── fantasy akan (akan) ──
  Page 1: 3 results | 3 new | total: 642

── fantasy zulu (zulu) ──
  Page 1: 9 results | 3 new | total: 645

── fantasy orisha (orisha) ──
  Page 1: 4 results | 3 new | total: 648

── fantasy anansi (anansi) ──
  Page 1: 19 results | 16 new | total: 664

── afrofuturism (afrofuturism) ──
  Page 1: 61 results | 58 new | total: 722

── africanfuturism (afrofuturism) ──
  Pag

In [11]:
# ── Audit: check what we're actually getting ───────────────────────────────
import json
from pathlib import Path

with open(CKPT_PATH) as f:
    books = json.load(f)

print(f"Total books so far: {len(books):,}")
print(f"\n── Sample book (full record) ──")
import pprint
pprint.pprint(books[0])

print(f"\n── Field coverage ──")
fields = ["title", "author", "year_published", "cover_url", "avg_rating", "num_ratings", "subjects", "source_url"]
for field in fields:
    filled = sum(1 for b in books if b.get(field) and b[field] != "" and b[field] != [] and b[field] is not None)
    pct = filled / len(books) * 100
    print(f"  {field:20s}: {filled:,}/{len(books):,} ({pct:.0f}%)")

print(f"\n── Source tag breakdown ──")
from collections import Counter
for tag, count in Counter(b["source_tag"] for b in books).most_common():
    print(f"  {tag:25s}: {count:,}")

Total books so far: 4,364

── Sample book (full record) ──
{'author': 'Edgar Rice Burroughs',
 'avg_rating': 4.4,
 'cover_url': 'https://covers.openlibrary.org/b/id/6608849-M.jpg',
 'description': '',
 'num_ratings': 5,
 'query': 'fantasy africa',
 'source': 'open_library',
 'source_tag': 'africa',
 'source_url': 'https://openlibrary.org/works/OL1418251W',
 'subjects': ['British in fiction',
              'Historical Fiction',
              'Wild men in fiction',
              'American Adventure stories',
              'Open Library Staff Picks',
              'American Fantasy fiction',
              'British',
              'Fiction',
              'Tarzan (Fictitious character)',
              'Adventure stories',
              'Wild men',
              'Africa in fiction',
              'Classic Literature',
              'Juvenile Fiction',
              'Tarzan (fictitious character), fiction',
              'Fiction, action & adventure',
              'Africa, fiction',
       

In [12]:
import json, pandas as pd
from pathlib import Path

REPO_ROOT = Path().resolve().parents[1]
RAW_DIR   = REPO_ROOT / "Data" / "Raw" / "non_western_fantasy"

# ── Load both ──────────────────────────────────────────────────────────────
ol = pd.DataFrame(json.load(open(RAW_DIR / "ol_genre_first.json")))
gr = pd.read_csv(RAW_DIR / "goodreads_with_descriptions.csv")

print(f"OL shape:         {ol.shape}")
print(f"Goodreads shape:  {gr.shape}")

# ── OL field coverage ──────────────────────────────────────────────────────
print("\n── OL field coverage ──")
for col in ["title","author","year_published","cover_url","avg_rating","num_ratings","subjects"]:
    filled = ol[col].apply(lambda x: bool(x) if not isinstance(x, float) else pd.notna(x)).sum()
    print(f"  {col:20s}: {filled:,}/{len(ol):,} ({filled/len(ol)*100:.0f}%)")

# ── Goodreads field coverage ───────────────────────────────────────────────
print("\n── Goodreads field coverage ──")
for col in ["title","author","description","cover_url","avg_rating","num_ratings","year_published"]:
    if col in gr.columns:
        filled = gr[col].notna().sum()
        print(f"  {col:20s}: {filled:,}/{len(gr):,} ({filled/len(gr)*100:.0f}%)")

# ── Goodreads shelf breakdown ──────────────────────────────────────────────
print("\n── Goodreads shelf breakdown ──")
print(gr["shelf"].value_counts().to_string())

# ── OL source_tag breakdown ────────────────────────────────────────────────
print("\n── OL source_tag breakdown ──")
print(ol["source_tag"].value_counts().to_string())

OL shape:         (4364, 12)
Goodreads shape:  (2129, 9)

── OL field coverage ──
  title               : 4,364/4,364 (100%)
  author              : 4,255/4,364 (98%)
  year_published      : 4,305/4,364 (99%)
  cover_url           : 2,446/4,364 (56%)
  avg_rating          : 759/4,364 (17%)
  num_ratings         : 759/4,364 (17%)
  subjects            : 3,899/4,364 (89%)

── Goodreads field coverage ──
  title               : 2,129/2,129 (100%)
  author              : 2,129/2,129 (100%)
  description         : 2,108/2,129 (99%)
  cover_url           : 2,129/2,129 (100%)
  avg_rating          : 1,996/2,129 (94%)
  num_ratings         : 1,983/2,129 (93%)
  year_published      : 1,713/2,129 (80%)

── Goodreads shelf breakdown ──
shelf
asian-fantasy              988
afrofuturism               741
african-fantasy            221
middle-eastern-fantasy      57
australian-fantasy          42
indigenous-fantasy          31
asian-science-fiction       23
latin-american-fantasy      17
south-ameri

In [ ]:
from langdetect import detect
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_BOOKS_API_KEY = os.getenv("GOOGLE_BOOKS_API_KEY", "")
print(f"API key loaded: {'✅' if GOOGLE_BOOKS_API_KEY else '❌ missing'}")
def fetch_google_books_description(title, author=""):
    query = f"{title} {author}".strip()
    url   = "https://www.googleapis.com/books/v1/volumes"
    params = {
        "q":          query,
        "maxResults": 5,  # get more results so we can find English one
        "key":        GOOGLE_BOOKS_API_KEY,
        "fields":     "items(volumeInfo(title,authors,description,publishedDate,imageLinks))"
    }

    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        items = data.get("items", [])
        if not items:
            return None

        # Try each result until we find an English description
        for item in items:
            info = item.get("volumeInfo", {})
            desc = info.get("description", "")
            if desc:
                try:
                    if detect(desc) == "en":
                        return {
                            "description": desc,
                            "cover_url":   info.get("imageLinks", {}).get("thumbnail", ""),
                            "published":   info.get("publishedDate", ""),
                        }
                except:
                    continue

        return None
    except:
        return None

# ── Quick test ─────────────────────────────────────────────────────────────
result = fetch_google_books_description("Children of Blood and Bone", "Tomi Adeyemi")
if result:
    print(f"Description: {result['description'][:200]}")
    print(f"Cover URL:   {result['cover_url']}")
else:
    print("No English description found")

In [ ]:
# ── Google Books enrichment loop ───────────────────────────────────────────
GB_CKPT_PATH = RAW_DIR / "google_books_checkpoint.json"

# Load checkpoint if exists
if GB_CKPT_PATH.exists():
    with open(GB_CKPT_PATH) as f:
        gb_ckpt = json.load(f)
    gb_results  = gb_ckpt["results"]
    gb_done     = set(gb_ckpt["done_keys"])
    print(f"Resuming — {len(gb_done):,} done")
else:
    gb_results = {}
    gb_done    = set()
    print(f"Starting fresh")

# Only enrich books still missing descriptions
needs_gb = [b for b in merged if not b.get("description") and b.get("title")]
remaining_gb = [b for b in needs_gb if b.get("ol_key", b.get("lt_url", "")) not in gb_done]

print(f"Books needing description: {len(needs_gb):,}")
print(f"Remaining to fetch:        {len(remaining_gb):,}")

stats = {"filled": 0, "not_found": 0, "error": 0}

for i, book in enumerate(tqdm(remaining_gb, desc="Google Books enrichment")):
    key    = book.get("ol_key") or book.get("lt_url", "")
    title  = book.get("title", "")
    author = book.get("author_name", [""])[0] if book.get("author_name") else ""

    try:
        result = fetch_google_books_description(title, author)
        if result and result["description"]:
            gb_results[key] = result
            stats["filled"] += 1
        else:
            gb_results[key] = {}
            stats["not_found"] += 1
    except Exception as e:
        gb_results[key] = {}
        stats["error"] += 1

    gb_done.add(key)

    if (i + 1) % 100 == 0:
        with open(GB_CKPT_PATH, "w") as f:
            json.dump({"results": gb_results, "done_keys": list(gb_done)}, f)
    
    time.sleep(0.5)

# Final checkpoint
with open(GB_CKPT_PATH, "w") as f:
    json.dump({"results": gb_results, "done_keys": list(gb_done)}, f)

print(f"\n✅ Google Books enrichment complete")
print(f"   Filled:    {stats['filled']:,}")
print(f"   Not found: {stats['not_found']:,}")
print(f"   Errors:    {stats['error']:,}")